In [ ]:

import requests
import pandas as pd
import json
import re

# Global Variables
URL_BASE = "https://ghoapi.azureedge.net/api/"

SADC = ["AGO", "NAM", "ZAF", "LSO", "SWZ", "BWA", "ZWE", "ZMB", "MOZ", "MWI", "MDG", "COM", "SYC", "MUS"]
EAC = ["BDI", "KEN", "RWA", "SSD", "TZA", "UGA", "COD"]
ECOWAS = ["BEN", "BFA", "CPV", "CIV", "GMB", "GHA", "GIN", "GNB", "LBR", "MLI", "NER", "NGA", "SEN", "SLE", "TGO"]
COUNTRY = SADC + EAC + ECOWAS

INDICATOR_TEXT = "tuberculosis"
INDICATOR_TEXT_TB = "tb"
INDICATOR = "TB_"

# Helper function to get JSON and convert to DataFrame
def convert_JSON_to_tbl(url):
    response = requests.get(url)
    data = response.json()
    if 'value' in data:
        return pd.json_normalize(data['value'])
    else:
        return pd.json_normalize(data)

# Fetch data
all_dimension = convert_JSON_to_tbl("https://ghoapi.azureedge.net/api/Dimension")
all_countries = convert_JSON_to_tbl(URL_BASE + "Dimension/COUNTRY/DimensionValues")[["Code", "Title"]]
all_indicators = convert_JSON_to_tbl(URL_BASE + "Indicator").drop(columns=["Language"], errors='ignore')

# Filter indicators
indicators_from_code = all_indicators[all_indicators["IndicatorCode"].str.contains(INDICATOR, case=False, na=False)]
indicators_from_text = all_indicators[all_indicators["IndicatorName"].str.contains(INDICATOR_TEXT, case=False, na=False)]
indicators_from_text_tb = all_indicators[all_indicators["IndicatorName"].str.contains(INDICATOR_TEXT_TB, case=False, na=False)]

indicators = pd.concat([indicators_from_code, indicators_from_text, indicators_from_text_tb]).drop_duplicates()
indicator_codes = indicators["IndicatorCode"].tolist()

# Download indicator data
indicator_data = []
for code in indicator_codes:
    url = URL_BASE + code
    response = requests.get(url)
    data = json.loads(response.text)
    if 'value' in data:
        df = pd.json_normalize(data['value'])
        df["IndicatorCode"] = code
        indicator_data.append(df)

# Combine all data
indicator_data_tbl = pd.concat(indicator_data, ignore_index=True)

# Final data cleaning and transformation
all_data = indicator_data_tbl[indicator_data_tbl["SpatialDim"].isin(COUNTRY)].copy()

all_data["region"] = all_data["SpatialDim"].apply(
    lambda x: "SADC" if x in SADC else "EAC" if x in EAC else "ECOWAS" if x in ECOWAS else None
)

all_data.rename(columns={
    "SpatialDim": "country_code",
    "TimeDim": "year",
    "NumericValue": "value",
    "High": "high_value",
    "Low": "low_value",
    "IndicatorCode": "indicator_code"
}, inplace=True)

final_data = all_data.merge(indicators, left_on="indicator_code", right_on="IndicatorCode", how="left")
final_data = final_data.merge(all_countries, left_on="country_code", right_on="Code", how="left")

final_data.rename(columns={
    "IndicatorName": "indicator_name",
    "Title": "country"
}, inplace=True)

final_data = final_data[[
    "indicator_code", "indicator_name", "year", "country_code", "country", 
    "value", "high_value", "low_value", "region"
]]

final_data.head()
